# AstroShield: evolucionar una constelación inteligente

**Fundamentos de Inteligencia Artificial · LITIID73**

Equipo: @AlexRecaman, @gretelZavaleta, @Henrrucho, @TITAFA

Sistema que usa un algoritmo evolutivo para colocar 8 satélites que observen seis regiones de la Tierra, respetando zonas de desechos espaciales y posiciones ocupadas, y que muestra el resultado en un globo 3D.

| Parte | Secciones | Responsable |
|---|---|---|
| 1 | §0 a §3: definición del problema, escenario, geometría y cobertura | @AlexRecaman |
| 2 | §4 a §6: función de calidad, algoritmo evolutivo y etapa 1 | @gretelZavaleta |
| 3 | §7 a §10: restricciones, etapa 2, cambio de condiciones y pruebas de restricciones | @Henrrucho |
| 4 | §11 a §15: globo 3D, variación entre ejecuciones, prueba propia, análisis y uso de IA | @TITAFA |

## §0. Cómo ejecutar

1. Abrir el notebook en Google Colab (*Archivo → Abrir cuaderno → GitHub*, repo `AlexRecaman/astroshield`).
2. *Entorno de ejecución → Ejecutar todo*. No requiere instalar nada: usa `numpy`, `pandas`, `matplotlib` y `plotly`, que ya vienen en Colab.
3. Las celdas deben ejecutarse en orden: cada sección usa funciones de las anteriores.

Las ejecuciones usan semillas fijas, así que se pueden repetir. Las cifras que se citan en el análisis se imprimen desde los resultados, no se escriben a mano.

## §1. Definición del problema (Entregable 1)

### 1.1 Supuestos del modelo

- La Tierra es una esfera de radio 6371 km. Las posiciones se expresan en latitud y longitud (grados).
- Cada satélite se modela por su **punto subsatélite**: el punto de la superficie que tiene justo debajo. No se simula movimiento orbital; la configuración es una "foto" de la constelación. El enunciado permite esta simplificación.
- Todos los satélites están a la misma altura y ven un círculo de la superficie (su **huella**) cuyo radio depende de esa altura y de la elevación mínima a la que el sensor puede observar.
- Las zonas de desechos y las posiciones ocupadas son círculos sobre la esfera. Restringen **dónde puede estar un satélite**, no lo que puede observar: un satélite fuera de una zona puede observar una región que esté debajo de ella.

### 1.2 Representación de una configuración

Una configuración es una matriz de **8 × 2**: una fila por satélite con su `(latitud, longitud)` en grados. Es el genoma del algoritmo evolutivo. El orden de los satélites no importa: dos matrices con las mismas filas en distinto orden son la misma constelación.

### 1.3 Cuándo una región está cubierta

- **Radio de la huella.** Para altura `h = 550 km` y elevación mínima `ε = 30°`, el ángulo central de la huella es
  `ρ = 90° − ε − asin(R·cos ε / (R + h))` ≈ **7.1°** (≈ 790 km sobre la superficie). Un punto está dentro de la huella si su distancia angular al punto subsatélite es ≤ ρ.
- **Área de la región.** Cada región es un disco de 300 km de radio (≈ 2.7°) alrededor de su centro. Se representa con **37 puntos**: el centro y tres anillos de 12 puntos a 1/3, 2/3 y 3/3 del radio.
- **Fracción cubierta `f`.** Proporción de esos 37 puntos que quedan dentro de al menos una huella.
- **Fracción redundante `d`.** Proporción de los puntos que quedan dentro de dos o más huellas.
- Una región está **cubierta** si `f ≥ 0.9`.

### 1.4 Calidad de una configuración

Las prioridades se traducen en pesos: **Alta = 3, Media = 2, Normal = 1** (suma 13 para las seis regiones). La calidad es

`F = Σ wᵣ·fᵣ / Σ wᵣ  +  β · Σ wᵣ·dᵣ / Σ wᵣ`,  con `β = 0.25`.

- El primer término (máximo 1) premia cubrir las regiones, más las de prioridad alta.
- El segundo premia la redundancia: con 8 satélites y 6 regiones sobran 2, y conviene usarlos para observar dos veces las regiones más importantes. `β` es pequeño para que la redundancia nunca valga más que cubrir una región nueva.
- Como ninguna pareja de regiones está a menos de 2·(ρ + 2.7°), ningún satélite puede cubrir dos regiones a la vez. El máximo alcanzable es entonces `1 + 0.25·(3 + 3)/13 ≈ 1.115`: todo cubierto y las dos regiones de prioridad Alta con doble cobertura. Ese valor sirve como referencia para medir qué tan cerca queda el algoritmo.

### 1.5 Configuraciones inválidas

Una configuración es inválida si algún satélite:

- está a una distancia angular menor que el radio de una de las tres **zonas de desechos** (Atlántico Norte 12°, Pacífico Sur 10°, polar norte 8°);
- está a menos de 5° de los **objetos A o B**;
- está a menos de **1°** de otro satélite de la constelación. Esta restricción la agrega el equipo para impedir dos satélites en la misma posición.

La **violación** de una configuración es la suma de cuántos grados invade cada satélite cada zona prohibida. Vale 0 si la configuración es válida.

### 1.6 Estrategia evolutiva

Algoritmo genético con codificación real:

- **Población inicial:** posiciones al azar distribuidas uniformemente sobre la esfera; con restricciones, solo posiciones válidas (muestreo con rechazo).
- **Selección:** torneo de 3 individuos.
- **Cruce uniforme por satélite:** el hijo toma cada satélite completo de uno de los dos padres. Se intercambian satélites, no coordenadas sueltas, para no romper posiciones que ya funcionan.
- **Mutación en dos escalas:** desplazamiento gaussiano de un satélite, la mitad de las veces grande (σ = 6°, explora) y la otra mitad fino (σ = 1.5°, ajusta). Con baja probabilidad, reubicación total del satélite en un punto al azar para escapar de óptimos locales. En pruebas preliminares con σ fijo de 6° el algoritmo se quedaba cerca del óptimo sin poder afinar la doble cobertura, porque para que dos satélites cubran por completo la misma región ambos deben quedar a menos de ≈ 4.4° de su centro.
- **Elitismo:** los 2 mejores pasan sin cambios a la siguiente generación.
- **Restricciones:** reparación (si una mutación deja un satélite en zona prohibida se busca la posición válida más cercana) y **regla de Deb** al comparar: una configuración válida siempre gana a una inválida, y entre inválidas gana la de menor violación. El resultado que se reporta es siempre la mejor configuración válida; si no hay ninguna, se informa.
- **Registro:** en cada generación se guarda la mejor, media y peor calidad, cuántos individuos son válidos y cuántas regiones cubre el mejor.

### 1.7 Parámetros y valores iniciales

| Parámetro | Valor inicial | Por qué |
|---|---|---|
| Satélites | 8 | Dato del problema |
| Altura / elevación mínima | 550 km / 30° | Órbita baja típica de observación; da una huella de ≈ 7° que alcanza para una región pero no para dos |
| Umbral de región cubierta | f ≥ 0.9 | Tolera que quede fuera el borde de la región |
| β (redundancia) | 0.25 | Menor que el peso de una región Normal (1/13 ≈ 0.077 frente a 0.25·3/13 ≈ 0.058 por región Alta duplicada) |
| Población / generaciones | 80 / 300 | Ajustado con 10 semillas: con 60 individuos y 40 generaciones de paciencia algunas corridas se quedaban con 5 regiones |
| Torneo | 3 | Presión de selección moderada |
| Prob. de cruce / mutación / reubicación | 0.9 / 0.2 por satélite / 0.05 | Valores habituales; se revisan con las pruebas |
| σ de mutación | 6° o 1.5° (50 % cada una) | 6° es del orden del radio de la huella; 1.5° permite afinar la doble cobertura |
| Elitismo | 2 | Conserva la mejor solución sin reducir la diversidad |
| Paciencia (parada) | 80 generaciones sin mejora | Corta ejecuciones estancadas sin detener las que aún mejoran despacio |

### 1.8 Cuándo una solución es suficientemente buena

- Las 6 regiones cubiertas (`f ≥ 0.9`).
- Las 2 regiones de prioridad Alta con redundancia.
- Violación 0 (cumple todas las restricciones).
- Calidad cercana al máximo de referencia (≈ 1.115).
- El algoritmo supera a una búsqueda aleatoria con el mismo número de evaluaciones.

Que el algoritmo deje de mejorar no prueba que encontró la mejor solución; por eso se compara con la referencia teórica, con varias semillas y con la búsqueda aleatoria.

## §2. Escenario

Los datos del problema se guardan como objetos editables. Un escenario reúne regiones, zonas prohibidas y parámetros del modelo, de modo que cambiar una condición (§9) es crear un escenario nuevo y volver a evaluar.

In [ ]:
import math
from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd

R_TIERRA_KM = 6371.0
PESOS_PRIORIDAD = {"Alta": 3, "Media": 2, "Normal": 1}


@dataclass(frozen=True)
class Region:
    nombre: str
    lat: float
    lon: float
    prioridad: str
    radio_km: float = 300.0


@dataclass(frozen=True)
class Zona:
    """Círculo donde no puede haber satélites. tipo: 'desechos' u 'ocupada'."""
    nombre: str
    lat: float
    lon: float
    radio_deg: float
    tipo: str


@dataclass
class Escenario:
    nombre: str
    regiones: list
    zonas: list
    n_satelites: int = 8
    altura_km: float = 550.0
    elevacion_min_deg: float = 30.0
    umbral_cubierta: float = 0.9
    beta_redundancia: float = 0.25
    separacion_min_deg: float = 1.0

    def pesos(self):
        return np.array([PESOS_PRIORIDAD[r.prioridad] for r in self.regiones], dtype=float)


REGIONES_BASE = [
    Region("Península de Yucatán", 20.5, -89.0, "Alta"),
    Region("Amazonía brasileña", -3.5, -62.0, "Alta"),
    Region("California", 36.8, -119.4, "Media"),
    Region("Sahel", 15.0, 0.0, "Media"),
    Region("Japón", 36.0, 138.0, "Media"),
    Region("Interior de Australia", -25.0, 133.0, "Normal"),
]

ZONAS_BASE = [
    Zona("Atlántico Norte", 45.0, -35.0, 12.0, "desechos"),
    Zona("Pacífico Sur", -30.0, -120.0, 10.0, "desechos"),
    Zona("Región polar norte", 78.0, 30.0, 8.0, "desechos"),
    Zona("Objeto A", 10.0, 45.0, 5.0, "ocupada"),
    Zona("Objeto B", -15.0, 110.0, 5.0, "ocupada"),
]


def escenario_base():
    return Escenario("Base", list(REGIONES_BASE), list(ZONAS_BASE))


pd.DataFrame([vars(r) | {"peso": PESOS_PRIORIDAD[r.prioridad]} for r in REGIONES_BASE])

In [ ]:
pd.DataFrame([vars(z) for z in ZONAS_BASE])

## §3. Geometría y modelo de cobertura

Funciones sobre la esfera, todas en grados:

- `distancia_angular`: ángulo central entre dos puntos (fórmula de haversine). Acepta arreglos de numpy.
- `radio_huella_deg`: radio angular de la huella a partir de la altura y la elevación mínima.
- `destino`: punto al que se llega desde otro avanzando cierta distancia angular con un rumbo dado. Sirve para muestrear regiones y dibujar círculos.
- `puntos_region`: los 37 puntos que representan una región.
- `fracciones_cobertura`: para una configuración, `f` y `d` de cada región. Los puntos de las regiones se calculan una sola vez y se reutilizan, porque el algoritmo evalúa miles de configuraciones.
- `cobertura_por_region`: la misma información en tabla, con el estado de cada región y la distancia al satélite más cercano.

In [ ]:
def distancia_angular(lat1, lon1, lat2, lon2):
    """Ángulo central en grados entre puntos (lat, lon) en grados. Admite arreglos."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return np.degrees(2 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0))))


def km_a_grados(km):
    return np.degrees(km / R_TIERRA_KM)


def radio_huella_deg(altura_km, elevacion_min_deg):
    eps = math.radians(elevacion_min_deg)
    eta = math.asin(R_TIERRA_KM * math.cos(eps) / (R_TIERRA_KM + altura_km))
    return 90.0 - elevacion_min_deg - math.degrees(eta)


def destino(lat, lon, dist_deg, rumbo_deg):
    """Punto a dist_deg de (lat, lon) siguiendo el rumbo (0 = norte, 90 = este)."""
    p1, l1 = math.radians(lat), math.radians(lon)
    d, th = math.radians(dist_deg), math.radians(rumbo_deg)
    p2 = math.asin(math.sin(p1) * math.cos(d) + math.cos(p1) * math.sin(d) * math.cos(th))
    l2 = l1 + math.atan2(math.sin(th) * math.sin(d) * math.cos(p1), math.cos(d) - math.sin(p1) * math.sin(p2))
    lon2 = (math.degrees(l2) + 180.0) % 360.0 - 180.0
    return math.degrees(p2), lon2


def puntos_region(region, anillos=3, por_anillo=12):
    """Centro y anillos concéntricos dentro del disco de la región: matriz (n, 2)."""
    radio = km_a_grados(region.radio_km)
    pts = [(region.lat, region.lon)]
    for k in range(1, anillos + 1):
        for j in range(por_anillo):
            pts.append(destino(region.lat, region.lon, radio * k / anillos, 360.0 * j / por_anillo))
    return np.array(pts)


def conteo_huellas(config, puntos, rho):
    """Cuántas huellas contienen a cada punto. config: (n_sat, 2); puntos: (m, 2)."""
    d = distancia_angular(puntos[:, None, 0], puntos[:, None, 1], config[None, :, 0], config[None, :, 1])
    return (d <= rho).sum(axis=1)


_CACHE_MUESTRAS = {}


def muestras_regiones(esc):
    """Puntos de todas las regiones apilados y el índice de región de cada punto (se calcula una vez)."""
    clave = tuple(esc.regiones)
    if clave not in _CACHE_MUESTRAS:
        pts = [puntos_region(r) for r in esc.regiones]
        _CACHE_MUESTRAS[clave] = (np.vstack(pts), np.repeat(np.arange(len(pts)), [len(p) for p in pts]))
    return _CACHE_MUESTRAS[clave]


def fracciones_cobertura(config, esc):
    """Arreglos f (cubierto por ≥1 huella) y d (por ≥2) con un valor por región."""
    pts, idx = muestras_regiones(esc)
    conteo = conteo_huellas(np.asarray(config, dtype=float), pts, radio_huella_deg(esc.altura_km, esc.elevacion_min_deg))
    n = len(esc.regiones)
    total = np.bincount(idx, minlength=n)
    f = np.bincount(idx, weights=conteo >= 1, minlength=n) / total
    d = np.bincount(idx, weights=conteo >= 2, minlength=n) / total
    return f, d


def cobertura_por_region(config, esc):
    """Tabla con f, d, estado de cada región y distancia al satélite más cercano."""
    config = np.asarray(config, dtype=float)
    f, d = fracciones_cobertura(config, esc)
    return pd.DataFrame([{
        "region": r.nombre,
        "prioridad": r.prioridad,
        "f": float(f[i]),
        "d": float(d[i]),
        "cubierta": bool(f[i] >= esc.umbral_cubierta),
        "dist_sat_mas_cercano": float(distancia_angular(r.lat, r.lon, config[:, 0], config[:, 1]).min()),
    } for i, r in enumerate(esc.regiones)])


RHO = radio_huella_deg(550.0, 30.0)
print(f"Radio de la huella: {RHO:.2f}° ≈ {math.radians(RHO) * R_TIERRA_KM:.0f} km")
print(f"Radio de una región (300 km): {km_a_grados(300):.2f}°")

### 3.1 Pruebas de la geometría

Antes de construir el algoritmo se comprueba que las piezas básicas dan resultados conocidos. Si alguna falla, todo lo que sigue estaría mal.

In [ ]:
# Distancias conocidas
assert abs(distancia_angular(0, 0, 0, 90) - 90) < 1e-9
assert abs(distancia_angular(0, 0, 0, 180) - 180) < 1e-9
assert abs(distancia_angular(90, 0, -90, 0) - 180) < 1e-9
assert abs(distancia_angular(20.5, -89.0, 20.5, -89.0)) < 1e-9
assert abs(distancia_angular(10, 179, 10, -179) - distancia_angular(10, 0, 10, 2)) < 1e-9  # cruza el antimeridiano

# Radio de la huella: a más altura o menos elevación mínima, huella más grande
assert 7.0 < RHO < 7.2
assert radio_huella_deg(800, 30) > RHO and radio_huella_deg(550, 10) > RHO

# destino() viaja exactamente la distancia pedida
for lat, lon, dist, rumbo in [(20.5, -89, 2.7, 45), (-25, 133, 7.1, 200), (36, 138, 30, 300)]:
    la, lo = destino(lat, lon, dist, rumbo)
    assert abs(distancia_angular(lat, lon, la, lo) - dist) < 1e-9

# Muestreo de regiones: 37 puntos, todos dentro del disco
for r in REGIONES_BASE:
    pts = puntos_region(r)
    assert pts.shape == (37, 2)
    assert distancia_angular(r.lat, r.lon, pts[:, 0], pts[:, 1]).max() <= km_a_grados(r.radio_km) + 1e-9

# Cobertura: un satélite en el centro cubre la región entera; uno a 20° no la toca
esc = escenario_base()
yuc = REGIONES_BASE[0]
lejos = destino(yuc.lat, yuc.lon, 20, 90)
cob = cobertura_por_region(np.array([[yuc.lat, yuc.lon], [lejos[0], lejos[1]]]), esc)
assert cob.loc[0, "f"] == 1.0 and cob.loc[0, "d"] == 0.0 and cob.loc[0, "cubierta"]
assert cob.loc[1:, "f"].eq(0).all()

# Dos satélites cerca del centro dan cobertura redundante completa
dos = np.array([destino(yuc.lat, yuc.lon, 1.0, 0), destino(yuc.lat, yuc.lon, 1.0, 180)])
assert cobertura_por_region(dos, esc).loc[0, "d"] == 1.0

# Ninguna pareja de regiones puede compartir satélite: están a más de 2·(ρ + radio)
centros = np.array([[r.lat, r.lon] for r in REGIONES_BASE])
sep = distancia_angular(centros[:, None, 0], centros[:, None, 1], centros[None, :, 0], centros[None, :, 1])
np.fill_diagonal(sep, np.inf)
print(f"Separación mínima entre centros de región: {sep.min():.1f}° (límite para compartir satélite: {2 * (RHO + km_a_grados(300)):.1f}°)")
assert sep.min() > 2 * (RHO + km_a_grados(300))

print("Pruebas de geometría: OK")